# eo-degrade · 阶段 0：EuroSAT 分类基线（TerraTorch + Prithvi）

**国内云 GPU 版**（AutoDL / 阿里云 / 腾讯云均可，JupyterLab 环境）

**目标**：跑通 TerraTorch 微调 Prithvi-EO-1.0，在 EuroSAT 上得到 mIoU 基准（里程碑 M1）。

**数据**：本 notebook 用 torchgeo 自动下载 EuroSAT（无需手动添加数据集）。

**预计运行时间**：20–40 分钟（T4 GPU）。

In [ ]:
!pip install -q terratorch torchgeo

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 型号:', torch.cuda.get_device_name(0))
    print('显存:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 数据准备（自动下载）

用 torchgeo 自动从 Zenodo 下载 EuroSAT（RGB 版，约 90MB，10 类共 27000 张），下载后自动解压为 `data/EuroSAT/2750/` 结构，TerraTorch 可直接读取。

> 如果下载慢，可手动下载 EuroSAT_RGB.zip 解压到 `data/EuroSAT/2750/` 后跳过本格。

In [ ]:
import os
from torchgeo.datasets import EuroSAT

os.makedirs('data', exist_ok=True)

# download=True 自动下载并解压到 data/EuroSAT/2750/
ds = EuroSAT(root='data', download=True)
print(f'EuroSAT 下载完成，共 {len(ds)} 张影像（10 类）')

# 验证目录结构
eurosat_dir = 'data/EuroSAT/2750'
classes = sorted([d for d in os.listdir(eurosat_dir) if os.path.isdir(os.path.join(eurosat_dir, d))])
print('类别:', classes)
print('每类样本数:', {c: len(os.listdir(os.path.join(eurosat_dir, c))) for c in classes})

In [ ]:
# 生成 TerraTorch 训练配置（YAML）
yaml_content = '''
model:
  class_path: terratorch.tasks.ClassificationTask
  init_args:
    model_args:
      backbone: prithvi_eo_v1_100
      backbone_pretrained: true
      num_frames: 1
      bands:
        - BLUE
        - GREEN
        - RED
      num_classes: 10
      head_dropout: 0.1
    model_factory: EncoderDecoderFactory
    loss: ce
    lr: 1.0e-4

data:
  class_path: terratorch.datamodules.EuroSATDataModule
  init_args:
    root: data
    batch_size: 32
    num_workers: 2
    bands:
      - BLUE
      - GREEN
      - RED
    train_transform:
      - class_path: albumentations.Resize
        init_args:
          height: 224
          width: 224
    val_transform:
      - class_path: albumentations.Resize
        init_args:
          height: 224
          width: 224
    test_transform:
      - class_path: albumentations.Resize
        init_args:
          height: 224
          width: 224

trainer:
  max_epochs: 5
  accelerator: auto
  devices: 1
  log_every_n_steps: 10
'''

with open('eurosat_prithvi.yaml', 'w') as f:
    f.write(yaml_content)
print('配置已写入 eurosat_prithvi.yaml')

## 训练

使用 TerraTorch CLI 微调 Prithvi-EO-1.0（无需手写训练循环）。

In [ ]:
import subprocess

result = subprocess.run(
    ['terratorch', 'fit', '--config', 'eurosat_prithvi.yaml'],
    capture_output=True, text=True
)

print('=== STDOUT (尾部) ===')
print(result.stdout[-3000:])
print('\n=== STDERR (尾部) ===')
print(result.stderr[-1500:])
print('\n退出码:', result.returncode)

In [ ]:
# 从训练日志提取验证指标
import re

stdout = result.stdout
metrics = {}
for key in ['val_miou', 'val_accuracy', 'val_loss', 'test_miou', 'test_accuracy']:
    matches = re.findall(rf'{key}\s*=\s*([0-9.]+)', stdout)
    if matches:
        metrics[key] = float(matches[-1])

if metrics:
    print('=== 里程碑 M1：EuroSAT 基准 ===')
    for k, v in metrics.items():
        print(f'{k}: {v:.4f}')
else:
    print('未从日志中解析到指标，请检查上方训练日志；指标名可能随版本变化。')
    print('可尝试的关键词: val/miou, val/accuracy, val/loss')

## 完成与下一步

✅ **里程碑 M1 达成**：EuroSAT 干净基准已产出。

下一步（阶段 A/B）：
1. 把基准结果记录到 `results/`（CSV）
2. 在 `degrade/` 库上叠加 GSD / SNR / MTF / 云遮挡退化，扫描响应曲线
3. 将本 notebook 与仓库同步推送到 GitHub

> ⚠️ **用完记得关机**（AutoDL 按小时计费，不关机会持续扣费）。
> 注意：TerraTorch API 随版本演进，若报错请参考官方示例
> https://github.com/terrastackai/terratorch/tree/main/examples